In [1]:
# Parameters
DB_PATH          = "../../../DB/oedb_refiner_1st.db"
BENCHMARK_PATH   = "../../../data/input_data/benchmark_trainingset.xlsx"
MATCHED_CSV_PATH = "./matched_participants.csv"
BENCHMARK_SHEET  = "participants"
NOTEGROUP_ID_MIN = 1
NOTEGROUP_ID_MAX = 23
MATCH_THRESHOLD  = 75

FIELDS = [
    "full_name", "place_of_origin", "municipality",
    "learning_route", "projectID_age", "session_identifier"
]

WEIGHTS = {
    "full_name":          45,
    "place_of_origin":    15,
    "municipality":       15,
    "learning_route":     15,
    "projectID_age":       9,
    "session_identifier":  1,
}

In [2]:
import sqlite3
import json
import pandas as pd
from rapidfuzz import fuzz

pd.reset_option("display.max_rows")
pd.reset_option("display.max_colwidth")

def load_etl(db_path, id_min, id_max):
    con = sqlite3.connect(db_path)
    df = pd.read_sql_query(
        """SELECT participantID, notegroupID, full_name, session_identifier,
                  place_of_origin, municipality, learning_route, projectID_age
           FROM participants
           WHERE notegroupID BETWEEN ? AND ?""",
        con, params=(id_min, id_max)
    )
    con.close()
    return df

def load_benchmark(xlsx_path, sheet, id_min, id_max):
    df = pd.read_excel(xlsx_path, sheet_name=sheet, dtype=str)
    df["notegroupID"] = df["notegroupID"].astype(int)
    df = df[df["notegroupID"].between(id_min, id_max)]
    return df  # keep all columns so participantID and others are available

etl = load_etl(DB_PATH, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)
bm  = load_benchmark(BENCHMARK_PATH, BENCHMARK_SHEET, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)

print("ETL records:      ", len(etl))
print("Benchmark records:", len(bm))

ETL records:       87
Benchmark records: 92


In [3]:
import re

def extract_age(projectID_age_str):
    """Extract age value from projectID_age JSON string."""
    if pd.isna(projectID_age_str) or not projectID_age_str:
        return None
    try:
        d = json.loads(projectID_age_str)
        return next(iter(d.values()))
    except (json.JSONDecodeError, StopIteration, TypeError):
        return None

def normalise_str(val):
    """Lowercase + strip, return None if empty."""
    if pd.isna(val) or str(val).strip() in ("", "None", "nan"):
        return None
    return str(val).strip().lower()

def normalise_learning_route(val):
    """Strip route/punctuation noise then normalise."""
    s = normalise_str(val)
    if s is None:
        return None
    s = re.sub(r'\b(route|routes)\b', '', s)
    s = re.sub(r'[-,\s]+', '', s)
    return s.strip() or None

def field_similarity(etl_val, bm_val, field):
    """
    Return similarity score 0-100 for a single field, or None if either side is null.
    None signals the field should be excluded from weighted average.
    """
    if field == "projectID_age":
        e = extract_age(etl_val)
        b = extract_age(bm_val)
        if e is None or b is None:
            return None
        try:
            return max(0.0, 1 - abs(float(e) - float(b)) / 10) * 100
        except (TypeError, ValueError):
            return None

    if field == "learning_route":
        e = normalise_learning_route(etl_val)
        b = normalise_learning_route(bm_val)
        if e is None or b is None:
            return None
        return fuzz.ratio(e, b)

    if field in ("full_name", "session_identifier"):
        e = normalise_str(etl_val)
        b = normalise_str(bm_val)
        if e is None or b is None:
            return None
        if field == "full_name":
            return (fuzz.token_sort_ratio(e, b) + fuzz.partial_ratio(e, b)) / 2
        return fuzz.token_sort_ratio(e, b)
    
    # place_of_origin, municipality
    e = normalise_str(etl_val)
    b = normalise_str(bm_val)
    if e is None or b is None:
        return None
    return fuzz.ratio(e, b)


def pair_score(etl_row, bm_row):
    """
    Compute weighted similarity score for an ETL-BM participant pair.
    Fields where either side is null are excluded; weight is redistributed.
    """
    total_weight = 0
    weighted_sum = 0.0
    for field, weight in WEIGHTS.items():
        sim = field_similarity(etl_row.get(field), bm_row.get(field), field)
        if sim is not None:
            weighted_sum += sim * weight
            total_weight += weight
    if total_weight == 0:
        return 0.0
    return weighted_sum / total_weight

In [4]:
def map_participants(etl_df, bm_df, threshold):
    """
    For each notegroupID, compute pairwise scores and greedily assign
    best matches above threshold (one-to-one constraint).
    Returns:
        matched   : list of (etl_idx, bm_idx, score)
        etl_only  : list of etl_idx  → FP rows
        bm_only   : list of bm_idx   → FN rows
    """
    matched  = []
    etl_only = []
    bm_only  = []

    all_ng_ids = sorted(set(etl_df["notegroupID"].unique()) | set(bm_df["notegroupID"].unique()))

    for ng_id in all_ng_ids:
        etl_ng = etl_df[etl_df["notegroupID"] == ng_id]
        bm_ng  = bm_df[bm_df["notegroupID"]  == ng_id]

        if bm_ng.empty:
            etl_only.extend(etl_ng.index.tolist())
            continue
        if etl_ng.empty:
            bm_only.extend(bm_ng.index.tolist())
            continue

        # Build score matrix
        scores = {}
        for ei in etl_ng.index:
            for bi in bm_ng.index:
                scores[(ei, bi)] = pair_score(etl_ng.loc[ei], bm_ng.loc[bi])

        # Greedy best-match assignment (highest score first)
        used_etl = set()
        used_bm  = set()
        for (ei, bi), score in sorted(scores.items(), key=lambda x: -x[1]):
            if score < threshold:
                break
            if ei in used_etl or bi in used_bm:
                continue
            matched.append((ei, bi, round(score, 2)))
            used_etl.add(ei)
            used_bm.add(bi)

        etl_only.extend([i for i in etl_ng.index if i not in used_etl])
        bm_only.extend( [i for i in bm_ng.index  if i not in used_bm])

    return matched, etl_only, bm_only


matched, etl_only, bm_only = map_participants(etl, bm, MATCH_THRESHOLD)

print(f"Matched pairs : {len(matched)}")
print(f"ETL-only (FP) : {len(etl_only)}")
print(f"BM-only  (FN) : {len(bm_only)}")

Matched pairs : 87
ETL-only (FP) : 0
BM-only  (FN) : 5


In [5]:
# Inspect matched pairs with scores
match_rows = []
for ei, bi, score in matched:
    match_rows.append({
        "notegroupID":        etl.loc[ei, "notegroupID"],
        "etl_participantID":  etl.loc[ei, "participantID"],
        "bm_participantID":   bm.loc[bi, "participantID"] if "participantID" in bm.columns else None,
        "etl_full_name":      etl.loc[ei, "full_name"],
        "bm_full_name":       bm.loc[bi, "full_name"] if "full_name" in bm.columns else None,
        "score":              score,
    })

pd.DataFrame(match_rows)

,notegroupID,etl_participantID,bm_participantID,etl_full_name,bm_full_name,score
0,1,8,1,Ahmad Ahmad,Ahmad Ahmad,100.0
1,1,9,2,Yasmine Ahmad,Yasmine Ahmad,100.0
2,1,10,3,Layla Hamliko,Layla Hamliko,100.0
3,1,11,4,Ahmad Noman,Ahmad Noman,100.0
4,1,12,5,Zaid Kurami,Zaid Kurami,100.0
...,...,...,...,...,...,...
82,22,96,95,Mokhilesa,Mokhilesa,100.0
83,22,97,96,Ilham Elmoussawi,Ilham Elmoussawi,100.0
84,22,98,97,Mohammad mahi hamdou,Mohammad mahi hamdou,100.0
85,23,99,98,Mustafa Aykut Alp Yılmaz,Mustafa Aykut Alp Yılmaz,100.0


In [6]:
# Inspect unmatched ETL rows (FP)
print("=== ETL-only (FP) ===")
display(etl.loc[etl_only, ["notegroupID", "participantID", "full_name"]])

# Inspect unmatched BM rows (FN)
print("\n=== BM-only (FN) ===")
bm_cols = [c for c in ["notegroupID", "participantID", "full_name"] if c in bm.columns]
display(bm.loc[bm_only, bm_cols])

=== ETL-only (FP) ===


,notegroupID,participantID,full_name



=== BM-only (FN) ===


,notegroupID,participantID,full_name
51,13,58,Afram
52,13,60,Khaled
53,13,61,Abdin
69,16,77,NaN
70,16,78,NaN


In [7]:
# Save matched pairs as CSV for use in evaluate_answers
csv_rows = []
for ei, bi, score in matched:
    csv_rows.append({
        "notegroupID":            etl.loc[ei, "notegroupID"],
        "etl_participantID":      etl.loc[ei, "participantID"],
        "bm_participantID":       bm.loc[bi, "participantID"] if "participantID" in bm.columns else None,
        "etl_full_name":          etl.loc[ei, "full_name"],
        "bm_full_name":           bm.loc[bi, "full_name"] if "full_name" in bm.columns else None,
        "etl_session_identifier": etl.loc[ei, "session_identifier"] if "session_identifier" in etl.columns else None,
        "bm_session_identifier":  bm.loc[bi, "session_identifier"] if "session_identifier" in bm.columns else None,
        "match_score":            score,
    })

matched_csv = pd.DataFrame(csv_rows)
matched_csv.to_csv(MATCHED_CSV_PATH, index=False)
print(f"Saved {len(matched_csv)} matched pairs to {MATCHED_CSV_PATH}")

Saved 87 matched pairs to ./matched_participants.csv
